In [52]:
import pandas as pd
import numpy as np
from unidecode import unidecode

SILVER_DATA_PATH = "/lakehouse/default/Files/silver_seguranca_publica/"

StatementMeta(, 885623f9-669b-4de4-8541-bdbad4dcfe1e, 54, Finished, Available, Finished, False)

In [53]:
def processar_tb_entorpecentes():

    df_entorpecentes = pd.read_csv(
        SILVER_DATA_PATH + "silver_tb_entorpecentes.csv",
        sep=";",
        dtype={
            "CEP": "string",
            "QTDE (GRAMAS)": "string",
            "DESCRICAO_APRESENTACAO": "string",
        },
    )

    df_entorpecentes.columns = df_entorpecentes.columns.str.strip().str.lower()

    df_entorpecentes.columns = (
        df_entorpecentes.columns.str.replace(")", "")
        .str.replace("(", "")
        .str.strip()
        .str.replace(" ", "_")
    )

    df_entorpecentes = df_entorpecentes.loc[
        df_entorpecentes["nome_municipio"].str.contains("osasco", na=False, case=False)
    ].copy()

    # # mesclar com tabela de apreensão

    df_apreensao_entorpecentes = pd.read_csv(
        SILVER_DATA_PATH + "silver_tb_apreensao_entorpecentes.csv",
        sep=";",
        dtype={"DESCRICAO_APRESENTACAO": "string"},
    )

    df_apreensao_entorpecentes.columns = (
        df_apreensao_entorpecentes.columns.str.lower().str.strip().str.replace(" ", "_")
    )

    df_apreensao_entorpecentes = df_apreensao_entorpecentes[
        ["num_bo", "descr_tipolocal", "descr_subtipolocal", "bairro"]
    ].copy()

    df_entorpecentes = df_entorpecentes.merge(
        df_apreensao_entorpecentes, on="num_bo", how="left", suffixes=("", "_apreensao")
    )

    df_entorpecentes['qtde_kg'] = df_entorpecentes['qtde_gramas_arred'] / 1000

    df_entorpecentes = (
        df_entorpecentes.groupby(
            ["ano_bo", "mes_estatistica", "descr_tipolocal_apreensao", "descr_toxico"],
            as_index=False,
        )
        .agg({"qtde_kg": "sum", "num_bo": "nunique"})
        .sort_values("qtde_kg", ascending=False)
    )

    df_entorpecentes = df_entorpecentes.rename(columns={"ano_bo": "ano"})

    return df_entorpecentes


df_entorpecentes = processar_tb_entorpecentes()

StatementMeta(, 885623f9-669b-4de4-8541-bdbad4dcfe1e, 55, Finished, Available, Finished, False)

/tmp/ipykernel_7095/1161586344.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_entorpecentes = pd.read_csv(
/tmp/ipykernel_7095/1161586344.py:28: DtypeWarning: Columns (7,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df_apreensao_entorpecentes = pd.read_csv(


In [54]:
df_armas = pd.read_csv(
    SILVER_DATA_PATH + "silver_tb_armas_apreendidas.csv",
    sep=";",
    dtype={"CEP": "string"},
)
df_armas.columns = df_armas.columns.str.lower().str.strip()
df_armas = df_armas.loc[
    df_armas["nome_municipio_circ"].str.contains("osasco", na=False, case=False)
].copy()
df_armas = df_armas.groupby(
    ["ano_bo", "mes_estatistica", "nome_municipio_circ", "desc_arma_fogo"], as_index=False
).size()
df_armas = df_armas.rename(columns={"ano_bo": "ano", "nome_municipio_circ": "cidade"})

StatementMeta(, 885623f9-669b-4de4-8541-bdbad4dcfe1e, 56, Finished, Available, Finished, False)

/tmp/ipykernel_7095/1569660412.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_armas = pd.read_csv(


In [55]:
def processar_tb_prisoes():
    df_prisoes = pd.read_csv(
        SILVER_DATA_PATH + "silver_tb_prisoes.csv",
        sep=";",
        dtype={"DESCRICAO_APRESENTACAO": "string", "CEP": "string"},
    )
    df_prisoes['MÊS ESTATISTICA'] = df_prisoes[['MÊS ESTATISTICA', 'MES_ESTATISTICA']].bfill(axis=1).iloc[:, 0]
    df_prisoes = df_prisoes.drop(columns="MES_ESTATISTICA")
    df_prisoes.columns = (
        df_prisoes.columns.str.lower().str.strip().str.replace(" ", "_")
    )
    df_prisoes.columns = [unidecode(col) for col in df_prisoes.columns]

    df_prisoes = df_prisoes.loc[
        df_prisoes["nome_municipio_circ"].str.contains("osasco", na=False, case=False)
    ].copy()

    df_prisoes = df_prisoes.groupby(
        [
            "ano_bo",
            "mes_estatistica",
            "descr_tipolocal",
            "descr_subtipolocal",
        ],
        as_index=False,
    ).size()

    # df_prisoes = df_prisoes.rename(columns={"ano_bo": "ano"})
    return df_prisoes


df_prisoes = processar_tb_prisoes()

StatementMeta(, 885623f9-669b-4de4-8541-bdbad4dcfe1e, 57, Finished, Available, Finished, False)

/tmp/ipykernel_7095/3321494763.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_prisoes = pd.read_csv(


In [56]:
def processar_dados_criminais():
    df_origem = pd.read_csv(
        SILVER_DATA_PATH + "silver_tb_dados_criminais.csv", sep=";"
    )
    df = df_origem.copy()
    df.columns = [unidecode(col) for col in df.columns]
    df['NOME_MUNICIPIO_CIRCUNSCRIÇÃO'] = df[['NOME_MUNICIPIO_CIRCUNSCRICAO']].bfill(axis=1).iloc[:, 0]
    df = df.drop(columns=["NOME_MUNICIPIO_CIRCUNSCRICAO"])
    df.columns = df.columns.str.strip().str.lower()
    df["natureza_apurada"] = [
        unidecode(natureza) for natureza in df["natureza_apurada"]
    ]
    df["data_ocorrencia_bo_datetime"] = pd.to_datetime(
        df["data_ocorrencia_bo"], errors="coerce"
    )
    df.columns = [unidecode(col) for col in df.columns]

    colunas = ["bairro", "nome_municipio_circunscricao", "nome_municipio"]
    for col in colunas:
        df[col] = df[col].astype(str).str.strip().str.upper()
        df[col] = [unidecode(c) for c in df[col]]

    df['nome_municipio'] = np.where(
        df['cidade'].isnull(), df['nome_municipio'], df['cidade']
    )

    df = df.loc[df['nome_municipio_circunscricao'] == 'OSASCO'].copy()

    df = df.groupby(
        [
            "ano_estatistica",
            "mes_estatistica",
            "nome_municipio_circunscricao",
            "bairro",
            "natureza_apurada"
        ],
        as_index=False
    ).size().rename(columns={"size": "quantidade_ocorrencias"})

    return df

df_dados_criminais = processar_dados_criminais()

StatementMeta(, 885623f9-669b-4de4-8541-bdbad4dcfe1e, 58, Finished, Available, Finished, False)

/tmp/ipykernel_7095/3216558234.py:2: DtypeWarning: Columns (3,6,9,10,13,14,15,16,17,18,19,25,26,27,28,29,30,31,32,33,34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  df_origem = pd.read_csv(


In [57]:
sdf_entorpecentes = spark.createDataFrame(df_entorpecentes)
sdf_armas = spark.createDataFrame(df_armas)
sdf_prisoes = spark.createDataFrame(df_prisoes)
sdf_dados_criminais = spark.createDataFrame(df_dados_criminais)

(
    sdf_entorpecentes.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_seg_publica_entorpecentes")
)
(
    sdf_armas.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_seg_publica_armas")
)
(
    sdf_prisoes.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_seg_publica_prisoes")
)
(
    sdf_dados_criminais.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_seg_publica_dados_criminais")
)


StatementMeta(, 885623f9-669b-4de4-8541-bdbad4dcfe1e, 59, Finished, Available, Finished, False)